# 🔬 The NeRF Geometry Lab
### Interactive Exploration of the "Hidden Math" Behind Protein AI

---

## 🎯 What You'll Learn

Most modern protein AI models (like **AlphaFold** and **trRosetta**) don't predict 3D coordinates directly. Instead, they predict **internal coordinates** (bond lengths, angles, and torsions) and then use an algorithm called **NeRF** (Natural Extension Reference Frame) to build the 3D model atom-by-atom.

**In this tutorial:**
1. 📐 Explore the **Z-Matrix** (internal coordinate representation)
2. 🎛️ Use interactive sliders to manipulate backbone torsions (φ and ψ)
3. 📊 Watch a **real-time Ramachandran plot** update as you change angles
4. 🗺️ Visualize **distance matrices** showing how local changes affect global structure

> **💡 Why This Matters**: Understanding internal coordinates is crucial for protein structure prediction, molecular dynamics, and protein design. This is the mathematical foundation of modern structural biology AI.

---

In [ ]:
# 🔧 Environment Detection & Setup
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🌐 Running in Google Colab')
    try:
        import synth_pdb
        print('   ✅ synth-pdb already installed')
    except ImportError:
        print('   📦 Installing synth-pdb and dependencies...')
        !pip install -q synth-pdb py3Dmol biotite
        print('   ✅ Installation complete')
    import plotly.io as pio
    pio.renderers.default = 'colab'
else:
    print('💻 Running in local Jupyter environment')
    sys.path.append(os.path.abspath('../../'))

print('✅ Environment configured!')

In [ ]:
import io

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import py3Dmol
from biotite.structure.io.pdb import PDBFile
from IPython.display import HTML, clear_output, display
from plotly.subplots import make_subplots

from synth_pdb import PeptideGenerator

gen = PeptideGenerator('ALA-ALA-ALA-ALA-ALA-ALA-ALA-ALA-ALA-ALA')
print('✅ NeRF Geometry Lab Ready!')
print('   Loaded: 10-residue polyalanine α-helix')

## 📚 Internal Coordinates: The Language of Protein Structure

### Z-Matrix Representation

Instead of Cartesian coordinates (x, y, z), proteins can be described using **internal coordinates**:

| Coordinate | Symbol | Description | Typical Range |
|------------|--------|-------------|---------------|
| **Bond Length** | r | Distance between bonded atoms | 1.0-1.5 Å |
| **Bond Angle** | θ | Angle between 3 consecutive atoms | 100-120° |
| **Dihedral Angle** | φ, ψ, ω | Rotation around bonds | -180° to +180° |

### The Backbone Dihedrals

For each residue *i*, we have three key angles:

$$\phi_i = \text{dihedral}(C_{i-1}, N_i, C\alpha_i, C_i)$$
$$\psi_i = \text{dihedral}(N_i, C\alpha_i, C_i, N_{i+1})$$
$$\omega_i = \text{dihedral}(C\alpha_i, C_i, N_{i+1}, C\alpha_{i+1})$$

## 🎮 Interactive NeRF Lab

Use the sliders below to adjust the $\phi$ and $\psi$ angles of the central residues. Observe how local changes propagate downstream and affect the global topology.

In [ ]:
# Interactive Widget Code (Simplified for readability)
phi_slider = widgets.FloatSlider(value=-60, min=-180, max=180, step=5, description='φ Angle:')
psi_slider = widgets.FloatSlider(value=-45, min=-180, max=180, step=5, description='ψ Angle:')
out = widgets.Output()

def update(change):
    with out:
        clear_output(wait=True)
        # Generate structure with new angles
        phi = [phi_slider.value] * 10
        psi = [psi_slider.value] * 10
        res = gen.generate(phi_list=phi, psi_list=psi)
        
        # Visualize
        view = py3Dmol.view(width=400, height=300)
        view.addModel(res.pdb, "pdb")
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        display(view.show())

phi_slider.observe(update, 'value')
psi_slider.observe(update, 'value')

display(widgets.VBox([phi_slider, psi_slider, out]))
update(None)

---

## ⚖️ The Ground Truth Challenge: Geometry vs. Physics

In synthetic biology and Protein AI, we face a fundamental question: **What is the "Ground Truth"?**

1.  **Geometric Truth (The Ideal)**: A structure built exactly at the Ramachandran centers (-60, -45 for helices). This is the **Label Intent**.
2.  **Physical Truth (The Reality)**: A structure that has been energy-minimized to resolve steric clashes. This is what you would see in a test tube.

Let's compare them.

In [ ]:
from synth_pdb.generator import generate_pdb_content, PeptideResult
from synth_pdb.geometry import calculate_rmsd, kabsch_superposition
import os

seq = "MVLSPADKTN"
IS_CI = os.getenv('GITHUB_ACTIONS') == 'true'

if IS_CI:
    print("🧪 CI ENVIRONMENT DETECTED: Running in high-speed validation mode.")

# 1. Geometric Ground Truth (Non-minimized)
res_geom = generate_pdb_content(sequence_str=seq, conformation="alpha", minimize_energy=False)
struct_geom = PeptideResult(res_geom).structure

# 2. Physical Reality (Minimized)
do_minimize = not IS_CI

if IS_CI:
    print("⚠️  [CI MODE] Bypassing OpenMM minimization to prevent runner hangs.")
else:
    print("🧬 Generating minimized structure (200 iterations)... ")

res_phys = generate_pdb_content(
    sequence_str=seq, 
    conformation="alpha", 
    minimize_energy=do_minimize, 
    minimization_max_iter=200
)
struct_phys = PeptideResult(res_phys).structure

# 3. Align and measure Strain
ca_geom = struct_geom[struct_geom.atom_name == "CA"].coord
ca_phys = struct_phys[struct_phys.atom_name == "CA"].coord

rot, trans = kabsch_superposition(ca_phys, ca_geom)
fitted_coords = (rot @ ca_phys.T).T + trans
strain_rmsd = calculate_rmsd(fitted_coords, ca_geom)

print(f"📏 Conformational Strain (RMSD): {strain_rmsd:.3f} Å")
print("   Low RMSD (< 0.5 Å) means the ideal geometry was already physically happy.")
print("   High RMSD means the physics engine had to 'warp' the intent to avoid clashes.")

### Why this matters for AI
If you train an AI on **minimized** data, you are inadvertently teaching it the biases of the **forcefield** (e.g., Amber14). By using the **Geometric Truth** as our label, we ensure the model learns the underlying relationship between sequence and the **Platonic Ideal** of the fold.

---

## 🎓 Key Insights

1. **Local Changes → Global Effects**: Changing one residue's angles affects the entire downstream structure
2. **Ramachandran Constraints**: Only certain φ/ψ combinations are sterically allowed
3. **Distance Patterns**: α-helices show characteristic i, i+4 contacts; β-sheets show long-range contacts
4. **NeRF Reconstruction**: This is exactly how AlphaFold and other AI models build 3D structures!

## 📖 Further Reading

- Jumper et al. (2021). "Highly accurate protein structure prediction with AlphaFold." *Nature* 596:583-589. [DOI: 10.1038/s41586-021-03819-2](https://doi.org/10.1038/s41586-021-03819-2)
- Parsons et al. (2005). "Practical conversion from torsion space to Cartesian space for in silico protein synthesis." *J Comput Chem* 26:1063-1068. [DOI: 10.1002/jcc.20237](https://doi.org/10.1002/jcc.20237)
- Ramachandran et al. (1963). "Stereochemistry of polypeptide chain configurations." *J Mol Biol* 7:95-99. [DOI: 10.1016/S0022-2836(63)80023-6](https://doi.org/10.1016/S0022-2836(63)80023-6)

---

<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;'>
    <h3>🎉 Lab Session Complete!</h3>
    <p>You've mastered internal coordinates and NeRF geometry!</p>
</div>